# Tabela `fact_pagamento` (silver)

Carrega `payment` do bronze (`nessie.bronze_sakila.payment`), remove
duplicatas, renomeia as colunas para o padrao em portugues e escreve como
`fact_pagamento` no schema `silver`. A escrita é
idempotente (`createOrReplace`), então reexecuções recriam a tabela sem
duplicar dados.

In [ ]:
SPARK_APP_NAME = "cria-fact-payment-silver"

from nucleo import cria_spark_session

spark = cria_spark_session(SPARK_APP_NAME)
spark

In [ ]:
TABELA_ORIGEM = "nessie.bronze_sakila.payment"
TABELA_DESTINO = "nessie.silver.fact_pagamento"

df = (
    spark.table(TABELA_ORIGEM)
    .dropDuplicates()
    .withColumnRenamed("payment_id", "pagamento_id")
    .withColumnRenamed("customer_id", "cliente_id")
    .withColumnRenamed("staff_id", "funcionario_id")
    .withColumnRenamed("rental_id", "locacao_id")
    .withColumnRenamed("payment_date", "data_pagamento")
    .withColumnRenamed("amount", "valor")
)
df.writeTo(TABELA_DESTINO).createOrReplace()

In [ ]:
n = spark.table(TABELA_DESTINO).count()
print(f"{TABELA_DESTINO}: {n} linhas")